# File Parsing in Python
### Extracting Information from PDB Files

**Author:** Omar Hamouda

**Environment:** Python 3.x | Jupyter Notebook | Anaconda

---
## 📌 Overview
This notebook covers how to read, search, and extract information
from text files — using Protein Data Bank (PDB) files as a real example.

## 📚 Table of Contents
1. Working with File Paths (os.path)
2. Reading Files (readlines)
3. Searching for Patterns (if)
4. Splitting Strings (split)
5. Casting Types
6. Line Numbers (enumerate)
7. Exercise: Extract Method & Temperature

In [1]:
# Auto-Fix Working Directory 
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
print("Working directory set to:", os.getcwd())

Working directory set to: /home/omar/biochemist-python_ORGANIZED


In [ ]:
# Import the os library to work with file paths and directories
import os

In [ ]:
# Get the current working directory (the folder Jupyter is running from)
print(os.getcwd())

In [ ]:
# List all files and folders in the current working directory
print(os.listdir())

In [ ]:
# List the contents of the PDB_files directory inside data/
os.listdir('data/PDB_files')

In [ ]:
# Build the file path in a cross-platform way
protein_file = os.path.join('data', 'PDB_files', '4eyr.pdb')
print(protein_file)

### Absolute vs. Relative Paths

A file path can be either **relative** or **absolute**:

- **Relative** — describes location from the current directory
  (e.g., `data/PDB_files/4eyr.pdb`). This is portable and works
  across different machines and operating systems.

- **Absolute** — gives the complete path from the filesystem root
  (e.g., `C:\Users\...\Desktop\biochemist-python\data\...`). This is
  specific to one machine.

We use `os.path.abspath()` to convert a relative path into an absolute one.

In [ ]:
# Get the absolute path of the protein file
abs_path = os.path.abspath(protein_file)
print(abs_path)

### Reading the File — Best Practice

To read a file in Python, the recommended approach is to use a
**context manager** (`with open(...)`) instead of the older
`open()` + `close()` pattern.

**Why?**
- The file is **automatically closed** when the block ends.
- It closes the file **even if an error occurs** inside the block.
- It produces **cleaner and safer** code.

We use `readlines()` to read the file into a list of strings —
one string per line.

In [ ]:
# Open the PDB file using a context manager
with open(protein_file, "r") as outfile:
    data = outfile.readlines()

# Sanity check: how many lines did we read?
print("Number of lines:", len(data))

# Peek at the first 3 lines
for line in data[:3]:
    print(line)

### Searching for the Ligand Information

PDB files use **keywords** at the start of each line to categorize
the information. For example:
- `HEADER` → general description
- `TITLE`  → structure title
- `HETNAM` → name of a bound ligand (drug, small molecule)

Here, we search for the `HETNAM` line to identify the drug
bound to the HIV-1 protease.

In [ ]:
# Search each line for the HETNAM keyword
for line in data:
    if 'HETNAM' in line:
        HETNAM_line = line
        print(HETNAM_line)

### Splitting the HETNAM Line

The `HETNAM` line is a single string containing multiple pieces
of information separated by spaces. To extract a specific piece
(the ligand abbreviation), we use the `split()` function, which
divides a string into a list of components.

The default delimiter is a space, and consecutive spaces are
ignored automatically.

In [ ]:
# Split the HETNAM line into a list of words
words = HETNAM_line.split()
print(words)

In [ ]:
# The ligand abbreviation is the second element (index 1)
abbrev = words[1]
print(abbrev)

### Extracting the Number of Protein Atoms

The PDB file contains a `REMARK 3` line stating the number of
protein atoms. Here, the value is separated from the description
by a **colon (`:`)**, so we use a custom delimiter in `split()`.

In [ ]:
# Search for the PROTEIN ATOMS line
for line in data:
    if 'PROTEIN ATOMS' in line:
        PROTEIN_ATOM_line = line
        words = PROTEIN_ATOM_line.split(':')    # Note the custom delimiter
        print(words)

In [ ]:
# The atom count is the second element (index 1)
atoms = words[1]
print(atoms)

### Casting & Calculation

Even though `atoms` looks like a number, it is actually a **string**
because `readlines()` returns every line as text.

To perform math, we must **cast** it to a numeric type (`float`).
This is a common and essential step in real data analysis —
any value read from a file is a string until proven otherwise.

In [ ]:
# Convert the string to a float
atoms = float(atoms)
print("Type after casting:", type(atoms))
print("Value:", atoms)

# Now divide by 2 (HIV-1 protease is a dimer)
atoms_per_monomer = atoms / 2
print("Atoms per monomer:", atoms_per_monomer)

### Finding Line Numbers with `enumerate()`

Some PDB information spans **multiple consecutive lines**. For example,
the protein sequence is given in a series of `SEQRES` lines.

To extract such blocks of information, we first need to know the
**line number** where the block starts. Python's `enumerate()`
gives us both the line number and the line content in one loop.

In [ ]:
# Find the line number where the SEQRES block starts
with open(protein_file, "r") as outfile:
    data = outfile.readlines()
    for linenum, line in enumerate(data):
        if 'SEQRES   1 A' in line:
            print(linenum, ':', line, sep='')

### Extracting the Full Protein Sequence

The protein sequence in a PDB file is split across multiple
`SEQRES` lines. Each line contains:
- The keyword `SEQRES`
- A running line number (1, 2, 3, ...)
- A chain identifier (A, B, ...)
- The total number of residues
- Up to 13 three-letter amino acid codes

To rebuild the full sequence, we loop through all `SEQRES` lines
and collect the amino acid codes.

In [ ]:
# Look at lines from 310 to 330 to understand the structure
for i in range(310, 330):
    print(data[i])

In [ ]:
# Extract all SEQRES lines and identify where the block ends
sequence_lines = []
for line in data:
    if line.startswith('SEQRES'):
        sequence_lines.append(line)

print("Number of SEQRES lines:", len(sequence_lines))
for i, line in enumerate(sequence_lines):
    print(f"Line {i}: {line.strip()[:40]}...")  # show first 40 chars

### Direct Access Using Line Numbers

Once we know the line number of a block, we can access any
specific line directly using its index — this avoids scanning
the entire file again for every query.

In [ ]:
# The SEQRES block starts at line 310
# Print the next 5 lines directly using their indices
for i in range(310, 315):
    print(data[i])

###  Extracting the Experimental Method & Temperature

Now we apply the same **search → split → extract** pattern
to two additional pieces of information from the same PDB file:

- **EXPERIMENT TYPE** — the method used to determine the structure
  (e.g., X-RAY DIFFRACTION, NMR, Cryo-EM).
- **TEMPERATURE** — the temperature (in Kelvin) at which the
  structure was determined.

This demonstrates how one pattern can be reused to extract
**any** piece of information from a structured file.

In [ ]:
# Search for the EXPERIMENT TYPE line
for line in data:
    if 'EXPERIMENT TYPE' in line:
        experiment_line = line
        print(experiment_line)

In [ ]:
# Split on the colon and extract the value
experiment_value = experiment_line.split(':')[1].strip()
print("Experiment type:", experiment_value)

In [ ]:
# Filter to get ONLY the temperature with "KELVIN" unit
for line in data:
    if 'TEMPERATURE' in line and 'KELVIN' in line:
        temperature_line = line
        print(temperature_line)

In [ ]:
# Split on the colon and extract the temperature value
temperature_value = temperature_line.split(':')[1].strip()

# Cast to float so we can do math
temperature_value = float(temperature_value)

print("Temperature (K):", temperature_value)

# Bonus: convert to Celsius
temperature_celsius = temperature_value - 273.15
print("Temperature (°C):", round(temperature_celsius, 2))

---
## 📝 Summary & Key Takeaways

In this notebook, we learned how to **parse structured text files**
using PDB (Protein Data Bank) files as a real-world example.
We analyzed the structure `4eyr.pdb` — an HIV-1 protease in complex
with the drug **Ritonavir**.

### What we covered:
- **File Paths** — used `os.path.join()` for cross-platform paths.
- **Reading Files** — used `with open()` + `readlines()` as best practice.
- **Searching Patterns** — used `if 'KEYWORD' in line` to find specific lines.
- **Splitting Strings** — used `split()` with default and custom delimiters.
- **Casting** — converted strings to `float` for calculations.
- **Line Numbers** — used `enumerate()` to find where blocks start.
- **Filtering with Multiple Conditions** — combined `and` to refine searches.
- **Direct Access** — retrieved specific lines using `data[index]`.

### Results from `4eyr.pdb`:

| Property | Value |
|----------|-------|
| **Ligand abbreviation** | RIT |
| **Ligand full name** | RITONAVIR |
| **Protein atoms (total)** | 1514 |
| **Atoms per monomer** | 757 |
| **SEQRES lines** | 16 (two chains: A and B) |
| **Experimental method** | X-RAY DIFFRACTION |
| **Temperature (K)** | 298.0 |
| **Temperature (°C)** | 24.85 |

---

## ⚠️ Important Notes

### On File Reading
- `readlines()` **can only be called once** per file object. If you need
  the contents again, re-open the file.
- Always use `with open(...)` — it closes the file automatically,
  even if an error occurs.

### On Casting
- **Every value read from a file is a `str`.** Even if it looks like a
  number, you must cast it (e.g., `float(...)`) before doing math.

### On Filtering
- When a keyword appears in multiple lines, refine your search with
  `and` — for example, `'TEMPERATURE' in line and 'KELVIN' in line`.

---

## 🔜 Next Steps
- Extract **coordinate data** (`ATOM` lines) for 3D visualization.
- Parse multiple PDB files in a loop to build a dataset.
- Use `pandas` for tabular PDB data analysis.
- Explore **Biopython** for advanced structure parsing.

---

## 🧬 Biological Context

**4EYR** is a crystal structure of HIV-1 protease — a key drug target
in antiretroviral therapy. The bound ligand, **Ritonavir**, is a
protease inhibitor used in HIV treatment. Structures like these
provide the atomic-level detail needed for rational drug design.